# Imports

In [2]:
import os
import pickle
import re
import shutil
import sys
sys.path.append(os.path.dirname(os.getcwd()))
from itertools import product
from scipy.stats import norm
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from matplotlib.backends.backend_pdf import PdfPages
from tools import load_npy, load_yaml_as_df, load_pkl, exist_metric, exist_stf_metric, inverse_stf_metrics, keep_split, is_full_group, load_metric_from_log

plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
plt.rc('font', family='Arial')
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.size'] = 10

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Varying seeds

## load data

In [4]:
root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/results_ML3/diff_seed'
exp_dirs = os.listdir(root)
exp_dirs = [os.path.join(root, exp_dir) for exp_dir in exp_dirs]

params = ['model', 'pred_len', 'data_id', 'learning_rate', 'inner_lr', 'meta_lr', 'rec_lambda', 'auxi_lambda', 'reg_lambda', 'lradj', 'train_epochs', 'patience', 'batch_size', 'auxi_batch_size', 'warmup_steps', 'meta_inner_steps', 'overlap_ratio', 'num_tasks', 'max_norm', 'auxi_loss', 'first_order', 'dropout', 'cycle', 'fix_seed', 'task_name']
metric_names = ['mse', 'mae', 'cov']

df = []
for exp_dir in exp_dirs:
    runned, setting_dir = exist_metric(exp_dir)
    if not runned:
        continue

    config = load_yaml_as_df(os.path.join(setting_dir, 'config.yaml'))
    metric = load_npy(os.path.join(setting_dir, 'metrics.npy'))
    result = config[params]
    if len(metric) == 6:
        log_metrics = load_metric_from_log(os.path.join(exp_dir, 'result_long_term_forecast.txt'))
        cov_loss = log_metrics['cov'] if log_metrics and 'cov' in log_metrics else np.inf
        result.loc[:, metric_names] = metric[1], metric[0], cov_loss
    else:
        result.loc[:, metric_names] = metric[1], metric[0], metric[2]
    result.loc[:, ['meta_type']] = config[['meta_type']] if 'meta_type' in config.columns else 'all'
    result.loc[:, ['exp_dir']] = exp_dir
    df.append(result)

df = pd.concat(df, ignore_index=True)
df.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)


save_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/stats_ML3'
os.makedirs(save_root, exist_ok=True)
df.to_csv(f"{save_root}/varying_seed.csv", index=False)


df.head(5)

,model,pred_len,data_id,learning_rate,inner_lr,meta_lr,rec_lambda,auxi_lambda,reg_lambda,lradj,train_epochs,patience,batch_size,auxi_batch_size,warmup_steps,meta_inner_steps,overlap_ratio,num_tasks,max_norm,auxi_loss,first_order,dropout,cycle,fix_seed,task_name,mse,mae,cov,meta_type,exp_dir
24,TQNet,96,ECL,0.0005,0.0005,0.0005,1.0,0.0,0.0001,type1,30,5,16,1024,20,1,0.15,5,1.0,MAE,1,0.0,168,2021,long_term_forecast,0.142434,0.236667,inf,all,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...
25,TQNet,96,ECL,0.0005,0.0005,0.0005,1.0,0.0,0.0001,type1,30,5,16,1024,20,1,0.15,5,1.0,MAE,1,0.0,168,2022,long_term_forecast,0.142711,0.236870,inf,all,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...
26,TQNet,96,ECL,0.0005,0.0005,0.0005,1.0,0.0,0.0001,type1,30,5,16,1024,20,1,0.15,5,1.0,MAE,1,0.0,168,2024,long_term_forecast,0.143252,0.237088,inf,all,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...
27,TQNet,96,ECL,0.0005,0.0005,0.0005,1.0,0.0,0.0001,type1,30,5,16,1024,20,1,0.15,5,1.0,MAE,1,0.0,168,2025,long_term_forecast,0.142734,0.236844,inf,all,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...
28,TQNet,96,ECL,0.0050,0.0050,0.0500,1.0,0.0,0.0000,type1,30,5,16,64,300,1,0.00,3,5.0,MSE,1,0.0,168,2021,long_term_forecast_meta_ml3,0.134940,0.228984,0.091569,all,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...


## preprocess

In [5]:
save_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/stats_ML3'
log_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/logs'
baselines = pd.read_csv(f'{log_root}/baselines_chosen.csv')
finetunes_best = pd.read_csv(f'{save_root}/finetune_best.csv')

model = 'TQNet'

base = baselines.copy()
base = base[
    (base.model == model) &
    (base.data_id.isin(['ECL', 'Weather']))
]
base['label'] = 'DF'
base['fix_seed'] = 2023


best = finetunes_best.copy()
best = best[
    (best.model == model) &
    (best.data_id.isin(['ECL', 'Weather']))
]
best['fix_seed'] = 2023
best['label'] = 'QDF'

df2 = df.copy()
df2 = df2[
    (df2.model == model) &
    (df2.data_id.isin(['ECL', 'Weather']))
]
df2['label'] = df2['task_name'].apply(lambda x: 'QDF' if "ml3" in x else 'DF')
min_mse_idx = df2.groupby(['data_id', 'pred_len', 'fix_seed', 'label'])['mse'].idxmin()
df2 = df2.loc[min_mse_idx]

columns = ['pred_len', 'data_id', 'fix_seed', 'mse', 'mae', 'label']
df_seed = pd.concat([base[columns], best[columns], df2[columns]], ignore_index=True)
df_seed['pred_len'] = df_seed['pred_len'].astype(int)
df_seed['fix_seed'] = df_seed['fix_seed'].astype(int)
df_seed['mse'] = df_seed['mse'].astype(float)
df_seed['mae'] = df_seed['mae'].astype(float)

dst_order = ['ECL', 'Weather']
df_seed['data_id'] = pd.Categorical(df_seed['data_id'], categories=dst_order, ordered=True)

seed_order = [2021, 2022, 2023, 2024, 2025]
df_seed['fix_seed'] = pd.Categorical(df_seed['fix_seed'], categories=seed_order, ordered=True)

label_order = ['QDF', 'DF']
df_seed['label'] = pd.Categorical(df_seed['label'], categories=label_order, ordered=True)

df_seed_avg = df_seed.groupby(['data_id', 'fix_seed', 'label']).mean(numeric_only=True).reset_index()
df_seed_avg['pred_len'] = 'Avg'
df_seed = pd.concat([df_seed, df_seed_avg], ignore_index=True)

df_seed.sort_values(by=['data_id', 'fix_seed', 'label', 'pred_len'], inplace=True)
df_seed['pred_len'] = df_seed['pred_len'].astype(str)


df_seed_mean_seed = df_seed.groupby(['data_id', 'label', 'pred_len']).mean(numeric_only=True).reset_index()
df_seed_std_seed = df_seed.groupby(['data_id', 'label', 'pred_len']).std(numeric_only=True).reset_index()

df_seed_agg = pd.merge(df_seed_mean_seed, df_seed_std_seed, on=['data_id', 'label', 'pred_len'], suffixes=('_mean', '_std'))
df_seed_agg['mse'] = df_seed_agg['mse_mean'].apply(lambda x: "{:.3f}".format(x)) + r'$_{\pm ' + df_seed_agg['mse_std'].apply(lambda x: "{:.3f}".format(x)) + r'}$'
df_seed_agg['mae'] = df_seed_agg['mae_mean'].apply(lambda x: "{:.3f}".format(x)) + r'$_{\pm ' + df_seed_agg['mae_std'].apply(lambda x: "{:.3f}".format(x)) + r'}$'
df_seed_agg = df_seed_agg[['data_id', 'label', 'pred_len', 'mse', 'mae']]

pred_len_order = ['96', '192', '336', '720', 'Avg']
df_seed_agg['pred_len'] = pd.Categorical(df_seed_agg['pred_len'], categories=pred_len_order, ordered=True)
df_seed_agg.sort_values(by=['data_id', 'label', 'pred_len'], inplace=True)

df_seed_agg

/tmp/ipykernel_1255756/367588208.py:50: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_seed_avg = df_seed.groupby(['data_id', 'fix_seed', 'label']).mean(numeric_only=True).reset_index()
/tmp/ipykernel_1255756/367588208.py:58: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_seed_mean_seed = df_seed.groupby(['data_id', 'label', 'pred_len']).mean(numeric_only=True).reset_index()
/tmp/ipykernel_1255756/367588208.py:59: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the

,data_id,label,pred_len,mse,mae
3,ECL,QDF,96,0.135$_{\pm 0.000}$,0.229$_{\pm 0.000}$
0,ECL,QDF,192,0.153$_{\pm 0.000}$,0.245$_{\pm 0.000}$
1,ECL,QDF,336,0.169$_{\pm 0.000}$,0.262$_{\pm 0.000}$
2,ECL,QDF,720,0.202$_{\pm 0.002}$,0.291$_{\pm 0.002}$
4,ECL,QDF,Avg,0.165$_{\pm 0.001}$,0.257$_{\pm 0.000}$
8,ECL,DF,96,0.143$_{\pm 0.000}$,0.237$_{\pm 0.000}$
5,ECL,DF,192,0.161$_{\pm 0.000}$,0.252$_{\pm 0.000}$
6,ECL,DF,336,0.178$_{\pm 0.000}$,0.270$_{\pm 0.000}$
7,ECL,DF,720,0.218$_{\pm 0.000}$,0.303$_{\pm 0.000}$
9,ECL,DF,Avg,0.175$_{\pm 0.000}$,0.265$_{\pm 0.000}$


## write to table

In [6]:
contents = []

for pl in ['96', '192', '336', '720', 'Avg']:
    line = f"{pl} "
    for data_id in ['ECL', 'Weather']:
        _df = df_seed_agg[(df_seed_agg.data_id == data_id) & (df_seed_agg.pred_len == pl)]
        for row in _df.itertuples():
            line += f"& {row.mse} & {row.mae} "
    line += r"\\"
    contents.append(line)
    if pl == '720':
        contents.append(r"\cmidrule(lr){1-9}")

print('\n'.join(contents))

96 & 0.135$_{\pm 0.000}$ & 0.229$_{\pm 0.000}$ & 0.143$_{\pm 0.000}$ & 0.237$_{\pm 0.000}$ & 0.160$_{\pm 0.001}$ & 0.203$_{\pm 0.001}$ & 0.160$_{\pm 0.001}$ & 0.203$_{\pm 0.001}$ \\
192 & 0.153$_{\pm 0.000}$ & 0.245$_{\pm 0.000}$ & 0.161$_{\pm 0.000}$ & 0.252$_{\pm 0.000}$ & 0.208$_{\pm 0.001}$ & 0.246$_{\pm 0.001}$ & 0.211$_{\pm 0.001}$ & 0.248$_{\pm 0.001}$ \\
336 & 0.169$_{\pm 0.000}$ & 0.262$_{\pm 0.000}$ & 0.178$_{\pm 0.000}$ & 0.270$_{\pm 0.000}$ & 0.264$_{\pm 0.001}$ & 0.287$_{\pm 0.001}$ & 0.266$_{\pm 0.001}$ & 0.289$_{\pm 0.001}$ \\
720 & 0.202$_{\pm 0.002}$ & 0.291$_{\pm 0.002}$ & 0.218$_{\pm 0.000}$ & 0.303$_{\pm 0.000}$ & 0.343$_{\pm 0.001}$ & 0.340$_{\pm 0.001}$ & 0.345$_{\pm 0.001}$ & 0.342$_{\pm 0.000}$ \\
\cmidrule(lr){1-9}
Avg & 0.165$_{\pm 0.001}$ & 0.257$_{\pm 0.000}$ & 0.175$_{\pm 0.000}$ & 0.265$_{\pm 0.000}$ & 0.244$_{\pm 0.001}$ & 0.269$_{\pm 0.001}$ & 0.246$_{\pm 0.001}$ & 0.271$_{\pm 0.001}$ \\
